# VLM grass classifier — prototype demo

Hosted vision-language model classifies roadside vegetation maintenance urgency as `baixa` | `média` | `alta`.

- **Prototype only** — not wired into FastAPI `/infer`.
- **Default model:** `gemma-4-26b-a4b-it` (override with `VLM_MODEL`).
- **Live mode:** needs `GOOGLE_API_KEY` in `services/ml/.env` (or the environment).
- **Fake mode:** when the key is missing, or `VLM_FAKE=1` — returns a deterministic stub (filename hints `baixa`/`media`/`alta`); no API call.
- **Shoulder focus:** the model must judge only the grass strip immediately next to the paved road / acostamento (`faixa junto à pista`), not distant embankments or background bush (common in Fernão Dias–style shots).

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from IPython.display import Image, display
from PIL import Image as PILImage

# Resolve package root whether the kernel cwd is services/ml or the notebook dir.
NOTEBOOK_DIR = Path.cwd().resolve()
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "verdia_ml").is_dir():
        ML_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find services/ml (pyproject.toml + src/verdia_ml)")

SRC = ML_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SAMPLES = ML_ROOT / "notebooks" / "samples"
ENV_PATH = ML_ROOT / ".env"


def load_dotenv_file(path: Path) -> None:
    """Minimal .env loader — never prints values."""
    if not path.is_file():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip("'\"")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv_file(ENV_PATH)

from verdia_ml.vlm import DEFAULT_MODEL, classify_image, resolve_model, use_fake_mode

key_present = bool((os.environ.get("GOOGLE_API_KEY") or "").strip())
fake = use_fake_mode()
model = resolve_model()

print(f"ml_root: {ML_ROOT}")
print(f"env_file: {ENV_PATH} ({'found' if ENV_PATH.is_file() else 'missing'})")
print(f"GOOGLE_API_KEY: {'set' if key_present else 'not set'}")
print(f"mode: {'FAKE' if fake else 'LIVE'}")
print(f"model: {model} (default {DEFAULT_MODEL})")

## Sample images

Local Street View screenshots for demo only (`dutra.png`, `fernao_dias_1.png`, `fernao_dias_2.png`). Gitignored — drop PNGs into `samples/` yourself. Judge the **shoulder strip** next to the pavement, not distant vegetation.

In [ ]:
SAMPLE_NAMES = ("dutra.png", "fernao_dias_1.png", "fernao_dias_2.png")
paths = [SAMPLES / name for name in SAMPLE_NAMES]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, (
    f"missing sample(s) in {SAMPLES}: {missing}. "
    "Drop local Street View screenshots there (gitignored)."
)

for path in paths:
    im = PILImage.open(path)
    print(f"{path.name} — {im.size[0]}×{im.size[1]} (shoulder / faixa junto à pista)")
    display(Image(filename=str(path), width=420))

## Classify

Pretty-prints each verdict: `classe`, self-reported `confianca_declarada`, `justificativa`, optional height.

In [ ]:
def pretty(verdict_dict: dict) -> str:
    return json.dumps(verdict_dict, ensure_ascii=False, indent=2)


if fake and not key_present:
    print(
        "No GOOGLE_API_KEY — running FAKE mode.\n"
        "Add the key to services/ml/.env (never commit it) for live Google AI Studio calls."
    )
elif fake:
    print("VLM_FAKE is set — skipping live API calls.")

results = []
for path in paths:
    print("=" * 60)
    print(path.name)
    display(Image(filename=str(path), width=280))
    verdict = classify_image(path)
    row = verdict.to_dict()
    row["path"] = path.name
    results.append(row)
    print(pretty(row))

print("=" * 60)
print(f"done — {len(results)} image(s), mode={'fake' if results and results[0].get('fake') else 'live'}")